<a id="setup"></a>
# Expérience CorBaMa — tagset fin, tokenisation SentencePiece

Même split que le notebook précédent (même seed), mais cette fois on entraîne un tokeniseur SentencePiece uniquement sur le texte du train split de CorBaMa — jamais sur le val ni le test, pour éviter toute fuite.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from bambara_pos_utils import *

import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

corbama_fine_path = Path.cwd().parent / "08_real_data_evaluation" / "corbama_corpus_fine.tsv"
corbama_fine_data = load_bambara_corpus(corbama_fine_path)
label2id_fine, id2label_fine = build_fine_tagset(corbama_fine_data)

# MEME seed que le notebook word-tokenizer : garantit exactement le même split
corbama_train, corbama_val, corbama_test = split_corpus(corbama_fine_data, seed=42)
print(f"Train : {len(corbama_train)} | Val : {len(corbama_val)} | Test : {len(corbama_test)}")

Corpus chargé : 166361 phrases.
Train : 133088 | Val : 16636 | Test : 16637


<a id="train-sp"></a>
# Entraînement de SentencePiece sur le train split de CorBaMa

In [2]:
raw_text_path = Path.cwd() / "corbama_train_raw.txt"
with open(raw_text_path, "w", encoding="utf-8") as f:
    for words, _ in corbama_train:
        f.write(" ".join(words) + "\n")

sp_prefix = Path.cwd() / "corbama_sp"
if not (Path.cwd() / "corbama_sp.model").is_file():
    spm.SentencePieceTrainer.train(
        input=str(raw_text_path),
        model_prefix=str(sp_prefix),
        vocab_size=1500,      # corpus petit (3711 phrases train) -> vocab plus modeste que Bayelemabaga (4000-6000)
        model_type="unigram",
        pad_id=0, unk_id=1, bos_id=-1, eos_id=-1,
    )
    print("SentencePiece (CorBaMa) entraîné.")
else:
    print("SentencePiece (CorBaMa) déjà présent, réutilisation.")

sp_cb = spm.SentencePieceProcessor(model_file=str(Path.cwd() / "corbama_sp.model"))
print("Taille du vocabulaire :", sp_cb.get_piece_size())

SentencePiece (CorBaMa) déjà présent, réutilisation.
Taille du vocabulaire : 1500


<a id="training"></a>
# Entraînement du modèle sous-mots (broadcasting) sur le tagset fin

In [3]:
hyperparams = {
    "learning_rate": 1e-3,
    "batch_size": 16,
    "hidden_dim": 128,
    "embedding_dim": 100,
}

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)
checkpoint_path = MODELS_DIR / "corbama_sp_finetagset.pth"

model_sp, history_sp, best_val_loss_sp = run_training(
    corbama_train, corbama_val,
    dataset_cls=BambaraSubwordDataset,
    dataset_args=(sp_cb, label2id_fine, "broadcast"),
    vocab_size=sp_cb.get_piece_size(),
    tagset_size=len(label2id_fine),
    hyperparams=hyperparams,
    device=device,
    patience=5, max_epochs=40,
    checkpoint_path=checkpoint_path,
)

Époque  1 | Train Loss : 0.2810 | Val Loss : 0.1585 | Val Acc : 95.11%
Époque  2 | Train Loss : 0.1391 | Val Loss : 0.1368 | Val Acc : 95.82%
Époque  3 | Train Loss : 0.1139 | Val Loss : 0.1285 | Val Acc : 96.08%
Époque  4 | Train Loss : 0.0983 | Val Loss : 0.1229 | Val Acc : 96.33%
Époque  5 | Train Loss : 0.0878 | Val Loss : 0.1215 | Val Acc : 96.38%
Époque  6 | Train Loss : 0.0790 | Val Loss : 0.1230 | Val Acc : 96.41%
Époque  7 | Train Loss : 0.0726 | Val Loss : 0.1241 | Val Acc : 96.40%
Époque  8 | Train Loss : 0.0673 | Val Loss : 0.1262 | Val Acc : 96.40%
Époque  9 | Train Loss : 0.0629 | Val Loss : 0.1265 | Val Acc : 96.41%
Arrêt anticipé à l'époque 9.


<a id="evaluation"></a>
# Évaluation sur le test set CorBaMa (tagset fin)

In [4]:
acc_sp, report_sp = evaluate_subword_model_word_level(
    model_sp, corbama_test, sp_cb, id2label_fine, "broadcast", device
)
print(f"Accuracy sous-mots, tagset fin, sur test CorBaMa : {acc_sp*100:.2f}%\n")
print(report_sp)

Accuracy sous-mots, tagset fin, sur test CorBaMa : 97.24%

              precision    recall  f1-score   support

       PUNCT       1.00      1.00      1.00     39359
         adj       0.90      0.86      0.88      2692
       adj/n       0.33      1.00      0.50         1
         adv       0.93      0.91      0.92      2301
       adv.p       0.89      0.79      0.84        42
        conj       0.97      0.98      0.97     10312
      conv.n       0.53      0.08      0.13       104
         cop       0.96      0.95      0.96      9844
         dtm       0.95      0.90      0.93      8032
        intj       0.92      0.90      0.91       733
        mrph       0.71      0.42      0.53        12
           n       0.96      0.97      0.97     54604
      n.prop       0.93      0.92      0.93      8787
       n/adj       0.00      0.00      0.00         2
         n/v       0.00      0.00      0.00         6
         num       0.99      0.99      0.99      5348
      onomat       0.8